# WeatherCam walkthrough

Re-scores two stored heads on validation data only: the 128-d metric head on the VeRi inner validation rows, and the weather head on DAWN validation. One seed (42), one frozen backbone (OpenCLIP ViT-B/16, laion2b_s34b_b88k). No test-set numbers appear here.

The embedding and head files under outputs/ are git-ignored and are not in the repository, so the code cells only run where those files exist. Run from the project root or from notebooks/. The saved outputs come from running the code cells in order with a small script, not from a Jupyter kernel.

In [1]:
import hashlib
import os
from pathlib import Path

import numpy as np

from weathercam.core.config import load_config

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

VERI = "outputs/embeddings/veri_vitb16_laion2b.npz"
DAWN = "outputs/embeddings/dawn_vitb16_laion2b.npz"
METRIC = "outputs/heads_metric/vitb16_laion2b_g2_metric.npz"
WEATHER = "outputs/heads_weather/vitb16_laion2b_weather.npz"
seed = load_config().seed
for path in (VERI, DAWN, METRIC, WEATHER):
    digest = hashlib.sha256(Path(path).read_bytes()).hexdigest()
    print(digest[:8], path)
print("seed", seed)

b4d2e245 outputs/embeddings/veri_vitb16_laion2b.npz
7578d8fc outputs/embeddings/dawn_vitb16_laion2b.npz
e1d95f5d outputs/heads_metric/vitb16_laion2b_g2_metric.npz
198281d3 outputs/heads_weather/vitb16_laion2b_weather.npz
seed 42


In [2]:
from weathercam.retrieval.innersplit import veri_inner_split

with np.load(VERI, allow_pickle=True) as archive:
    veri = {key: archive[key] for key in archive.files}
fit_rows, val_rows = veri_inner_split(veri, seed)
print("fit images", len(fit_rows), "validation images", len(val_rows))
print("validation vehicles", len(set(veri["vehicle_ids"][val_rows].tolist())))

fit images 29918 validation images 7828
validation vehicles 119


In [3]:
from weathercam.retrieval.evaluate import camera_aware_scores, overall_score, query_scores
from weathercam.retrieval.index import BruteForceIndex
from weathercam.service.heads import load_metric_head

stored = np.load(METRIC, allow_pickle=False)
features = load_metric_head(METRIC).project(veri["embeddings"][val_rows])
vehicles = veri["vehicle_ids"][val_rows]
cameras = veri["camera_ids"][val_rows]
index = BruteForceIndex(features.shape[1])
index.add(features)
plain = overall_score(query_scores(index, features, vehicles))
camera = overall_score(camera_aware_scores(index, features, vehicles, cameras))
rows = (
    ("plain P@1", plain.precision_at_1, "val_plain_p1"),
    ("plain mAP@R", plain.map_at_r, "val_plain_map"),
    ("camera-aware P@1", camera.precision_at_1, "val_camera_p1"),
    ("camera-aware mAP@R", camera.map_at_r, "val_camera_map"),
)
for name, got, key in rows:
    print(name, "got", round(got, 6), "stored", round(float(stored[key]), 6))

plain P@1 got 0.962826 stored 0.962826
plain mAP@R got 0.478883 stored 0.478883
camera-aware P@1 got 0.816684 stored 0.816684
camera-aware mAP@R got 0.432795 stored 0.432795


In [4]:
from weathercam.attributes.weather import load_weather, part_arrays
from weathercam.core.cls_metrics import macro_f1
from weathercam.service.handlers import weather
from weathercam.service.heads import load_linear_head

head = load_linear_head(WEATHER, names_key="weathers")
x, y = part_arrays(load_weather(DAWN), "val")
truth = y.tolist()
predicted = [item.class_id for item in weather(head, x)]
correct = sum(int(a == b) for a, b in zip(truth, predicted))
print("validation images", len(truth), "correct", correct)
print("macro-F1", round(macro_f1(truth, predicted, list(range(len(head.class_ids)))), 6))

validation images 161 correct 127
macro-F1 0.788897


## Left out

- kNN out-of-distribution scoring: about 17 minutes per validation run here.
- The embedding and plate worker processes.
- Text-to-image search quality: it was never measured.
- Plate reading: weak off the training distribution, so no accuracy claim is made here.
- The weather head covers four DAWN classes only, and the VeRi heads are trained on VeRi.